In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm

from fmri_fm_eval.datasets.registry import create_dataset

In [2]:
DATASET_NAMES = {
    "abide_dx": "ABIDE Dx",
    # "abide_age": "ABIDE Age",
    # "abide_sex": "ABIDE Sex",
    "adhd200_dx": "ADHD200 Dx",
    # "adhd200_sex": "ADHD200 Sex",
    "adni_ad_vs_cn": "ADNI Dx",
    # "adni_sex": "ADNI Sex",
    "ppmi_dx": "PPMI Dx",
    # "ppmi_sex": "PPMI Sex",
    # "ppmi_age": "PPMI Age",
    # "aabc_sex": "HCP-A Sex",
    "aabc_age": "HCP-A Age",
    "hcpya_rest1lr_gender": "HCP-YA Sex",
    # "hcpya_rest1lr_age": "HCP-YA Age",
    # "hcpya_rest1lr_neofacn": "HCP-YA NEO-N",
    "hcpya_task21": "HCP-YA Task21",
    "nsd_cococlip": "NSD COCO24",
}

In [ ]:
CLASS_LABEL_LUT = {
    "abide_dx": {"Autism": "ASD", "Control": "Control"},
    "adni_ad_vs_cn": {1: "AD", 0: "Control"},
    "aabc_age": {0: "Q1", 1: "Q2", 2: "Q3", 3: "Q4"},
}

CASE_LABELS = {
    "abide_dx": "ASD",
    "adhd200_dx": "ADHD",
    "adni_ad_vs_cn": "AD",
    "ppmi_dx": "PD",
}

TASK_LUT = {
    "abide_dx": "ASD Dx",
    "adhd200_dx": "ADHD Dx",
    "adni_ad_vs_cn": "AD Dx",
    "ppmi_dx": "PD Dx",
}

CITATION_LUT = {
    "abide_dx": r"\cite{Di-Martino2014}",
    "adhd200_dx": r"\cite{ADHD-200-Consortium2012}",
    "adni_ad_vs_cn": r"\cite{Jack-Jr2008}",
    "ppmi_dx": r"\cite{Marek2011}",
    "aabc_age": r"\cite{Bookheimer2019}",
    "hcpya_rest1lr_gender": r"\cite{Van-Essen2013}",
    "hcpya_task21": r"\cite{Van-Essen2013}",
    "nsd_cococlip": r"\cite{Allen2022}",
}

STATE_DATASETS = {
    "hcpya_task21",
    "nsd_cococlip",
}

# train fraction from logistic probe
# (nb this was prob too small)
TRAIN_FRACTION = 0.7

In [25]:
def get_dataset_info(name):
    # load schaefer400 bc smallest
    dataset_dict = create_dataset(name, "schaefer400")

    splits = ["train", "validation", "test"]

    split_samples = {split: len(ds) for split, ds in dataset_dict.items()}
    all_samples = sum(split_samples.values())

    if name in STATE_DATASETS:
        # for state we used the fixed test split
        test_samples = split_samples["test"]
    else:
        # for trait, used looped logistic probe with random test splits
        test_samples = int((1 - TRAIN_FRACTION) * all_samples)
    samples_fmt = f"{all_samples} ({test_samples})"

    split_subs = {}
    split_sub_counts = {}
    for split, ds in dataset_dict.items():
        subs = np.unique(np.asarray(ds.dataset["sub"])[ds.indices])
        split_subs[split] = subs
        split_sub_counts[split] = len(subs)
    for ii, split_a in enumerate(splits):
        for split_b in splits[ii + 1 :]:
            overlap = len(np.intersect1d(split_subs[split_a], split_subs[split_b]))
            if overlap > 0:
                print(f"warning: {name} {split_a} {split_b} subject overlap {overlap}")
    all_sub_counts = len(np.unique(np.concatenate(list(split_subs.values()))))
    if name in STATE_DATASETS:
        test_sub_counts = split_sub_counts["test"]
    else:
        test_sub_counts = int((1 - TRAIN_FRACTION) * all_sub_counts)
    subs_fmt = f"{all_sub_counts} ({test_sub_counts})"

    sample_lengths = []
    sample_trs = []
    for split in splits:
        for sample in dataset_dict[split]:
            bold = sample["bold"]
            tr = sample["tr"]
            sample_lengths.append(len(bold))
            sample_trs.append(tr)
    sample_length = np.mean(sample_lengths)
    sample_tr = np.mean(sample_trs)

    label_counts = {}
    for split in splits:
        for label, count in zip(dataset_dict[split].labels, dataset_dict[split].label_counts):
            label_counts[label] = label_counts.get(label, 0) + count
    if name in CLASS_LABEL_LUT:
        label_counts = {
            label: label_counts[target] for target, label in CLASS_LABEL_LUT[name].items()
        }
    n_classes = len(label_counts)
    majority_ratio = max(label_counts.values()) / sum(label_counts.values())

    if name in STATE_DATASETS:
        label_counts_fmt = "-"
    else:
        label_counts_fmt = ", ".join(f"{label} ({count})" for label, count in label_counts.items())

    record = {
        "subjects": subs_fmt,
        "samples": samples_fmt,
        "sample length": f"{sample_length:.0f}",
        "sample tr": f"{sample_tr:.1f}s",
        "num classes": n_classes,
        "majority ratio": float(majority_ratio),
        "labels": label_counts_fmt,
    }
    return record

In [26]:
# dataset = create_dataset("abide_age", "schaefer400")
# print(dataset)
get_dataset_info("abide_dx")
# get_dataset_info("hcpya_task21")

{'subjects': '826 (247)',
 'samples': '826 (247)',
 'sample length': '150',
 'sample tr': '2.0s',
 'num classes': 2,
 'majority ratio': 0.5508474576271186,
 'labels': 'ASD (371), Control (455)'}

In [29]:
dataset_table = []
for key, label in tqdm(DATASET_NAMES.items()):
    name, target = label.split()
    if key in TASK_LUT:
        target = TASK_LUT[key]
    info = get_dataset_info(key)
    citation = CITATION_LUT[key]

    record = {"name": f"{name} \\scriptsize{{{citation}}}", "target": target, **info}
    dataset_table.append(record)

dataset_table = pd.DataFrame.from_records(dataset_table)
print(dataset_table.to_markdown(index=False))

100%|██████████| 8/8 [01:32<00:00, 11.60s/it]

| name                                                | target   | subjects   | samples      |   sample length | sample tr   |   num classes |   majority ratio | labels                                 |
|:----------------------------------------------------|:---------|:-----------|:-------------|----------------:|:------------|--------------:|-----------------:|:---------------------------------------|
| ABIDE \scriptsize{\cite{Di-Martino2014}}            | ASD Dx   | 826 (247)  | 826 (247)    |             150 | 2.0s        |             2 |        0.550847  | ASD (371), Control (455)               |
| ADHD200 \scriptsize{\cite{ADHD-200-Consortium2012}} | ADHD Dx  | 430 (129)  | 430 (129)    |             150 | 2.0s        |             2 |        0.565116  | ADHD (187), Control (243)              |
| ADNI \scriptsize{\cite{Jack-Jr2008}}                | AD Dx    | 410 (123)  | 410 (123)    |             100 | 3.0s        |             2 |        0.765854  | AD (96), Control (314)    

In [31]:
dataset_table_fmt = dataset_table.copy()
dataset_table_fmt["majority ratio"] = [
    f"{100 * ratio:.0f}\%" for ratio in dataset_table_fmt["majority ratio"]
]
dataset_table_fmt = dataset_table_fmt.iloc[:, :8]
dataset_table_fmt.columns = [
    "Dataset",
    "Target",
    "Subjects",
    "Samples",
    "Seq length",
    "TR",
    "\#Classes",
    "Majority class \%",
]
print(dataset_table_fmt.to_latex(index=False))

\begin{tabular}{llllllrl}
\toprule
Dataset & Target & Subjects & Samples & Seq length & TR & \#Classes & Majority class \% \\
\midrule
ABIDE \scriptsize{\cite{Di-Martino2014}} & ASD Dx & 826 (247) & 826 (247) & 150 & 2.0s & 2 & 55\% \\
ADHD200 \scriptsize{\cite{ADHD-200-Consortium2012}} & ADHD Dx & 430 (129) & 430 (129) & 150 & 2.0s & 2 & 57\% \\
ADNI \scriptsize{\cite{Jack-Jr2008}} & AD Dx & 410 (123) & 410 (123) & 100 & 3.0s & 2 & 77\% \\
PPMI \scriptsize{\cite{Marek2011}} & PD Dx & 662 (198) & 662 (198) & 120 & 2.5s & 2 & 62\% \\
HCP-A \scriptsize{\cite{Bookheimer2019}} & Age & 560 (168) & 560 (168) & 500 & 0.7s & 4 & 27\% \\
HCP-YA \scriptsize{\cite{Van-Essen2013}} & Sex & 598 (179) & 598 (179) & 500 & 0.7s & 2 & 50\% \\
HCP-YA \scriptsize{\cite{Van-Essen2013}} & Task21 & 614 (110) & 28071 (5040) & 16 & 1.0s & 21 & 17\% \\
NSD \scriptsize{\cite{Allen2022}} & COCO24 & 8 (1) & 48534 (5390) & 16 & 1.0s & 24 & 7\% \\
\bottomrule
\end{tabular}



In [34]:
import json

chance_accs = {}
for ii, key in enumerate(DATASET_NAMES):
    chance_accs[key] = round(float(dataset_table.iloc[ii]["majority ratio"]), 3)
print(json.dumps(chance_accs, indent=4))

{
    "abide_dx": 0.551,
    "adhd200_dx": 0.565,
    "adni_ad_vs_cn": 0.766,
    "ppmi_dx": 0.616,
    "aabc_age": 0.273,
    "hcpya_rest1lr_gender": 0.5,
    "hcpya_task21": 0.169,
    "nsd_cococlip": 0.068
}


In [4]:
for key in DATASET_NAMES:
    dataset = create_dataset(key, "schaefer400")

    split_subs = {}
    for split, ds in dataset.items():
        subs = np.asarray(ds.dataset["sub"])[ds.indices]
        subs, counts = np.unique(subs, return_counts=True)
        if counts.max() > 1:
            print(key, split, counts.max().item())
        split_subs[split] = subs

    splits = list(split_subs)
    for ii, split_a in enumerate(splits):
        for split_b in splits[ii + 1 :]:
            n_overlap = len(np.intersect1d(split_subs[split_a], split_subs[split_b]))
            if n_overlap > 0:
                print(key, split_a, split_b, n_overlap)

hcpya_task21 train 49
hcpya_task21 validation 49
hcpya_task21 test 49
nsd_cococlip train 6236
nsd_cococlip validation 5418
nsd_cococlip test 5390
nsd_cococlip testid 885
nsd_cococlip train testid 6
